## Linear Probes Experiment Runner

This notebook runs the new `linear_probes` pipeline from `utility_analysis/experiments/linear_probes/run_linear_probes.py`.

Flow:
1. Configure model/options/roles/utilities
2. Run `linear_probes_collect` via `run_experiments.py`
3. Run `linear_probes_train` via `run_experiments.py`
4. Inspect and plot per-layer probe performance

In [ ]:
import os
import json
import yaml
import subprocess
from pathlib import Path

# -----------------------------
# Colab bootstrap: clone repo into /content
# -----------------------------
REPO_URL = "https://github.com/thowell332/llm-preferences.git"
REPO_ROOT = Path("/content/llm_preferences").expanduser()
GIT_REF = "main"

if not REPO_ROOT.exists():
    !git clone {REPO_URL} {str(REPO_ROOT)}
else:
    print("Repo already exists at:", REPO_ROOT)

%cd {str(REPO_ROOT)}
!git fetch --all -q
!git reset --hard origin/{GIT_REF}

repo_root = REPO_ROOT
utility_dir = repo_root / "utility_analysis"

assert utility_dir.exists(), f"Expected utility_analysis/ under {repo_root}"
print("Repo root:", repo_root)
print("Utility dir:", utility_dir)

In [ ]:
# Install dependencies (Colab-friendly pins).
%pip -q install --upgrade pip
%pip -q install -r /content/llm_preferences/requirements.txt "jedi>=0.16" "rich<14"

print("Installed requirements.txt (+ Colab compatibility pins)")

In [ ]:
# Download selected model locally and patch utility_analysis/models.yaml.
from huggingface_hub import snapshot_download

MODEL_KEY_FOR_DOWNLOAD = "llama-31-8b-instruct"
MODELS_PATH = utility_dir / "models.yaml"

with open(MODELS_PATH, "r") as f:
    models = yaml.safe_load(f)

assert MODEL_KEY_FOR_DOWNLOAD in models, f"{MODEL_KEY_FOR_DOWNLOAD} not found in models.yaml"
model_entry = models[MODEL_KEY_FOR_DOWNLOAD]
HF_REPO_ID = model_entry.get("model_name")
if HF_REPO_ID is None:
    raise ValueError(f"Model entry for {MODEL_KEY_FOR_DOWNLOAD} missing model_name")

MODEL_DIR = Path(f"/content/models/{MODEL_KEY_FOR_DOWNLOAD}").expanduser()
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"Downloading {HF_REPO_ID} -> {MODEL_DIR}")
snapshot_download(
    repo_id=HF_REPO_ID,
    local_dir=str(MODEL_DIR),
    local_dir_use_symlinks=False,
    resume_download=True,
)

# Patch this model entry to use local path in Colab.
models[MODEL_KEY_FOR_DOWNLOAD]["path"] = str(MODEL_DIR)
models[MODEL_KEY_FOR_DOWNLOAD]["model_name"] = HF_REPO_ID

with open(MODELS_PATH, "w") as f:
    yaml.safe_dump(models, f, sort_keys=False)

print(f"Updated {MODELS_PATH} for {MODEL_KEY_FOR_DOWNLOAD} -> {MODEL_DIR}")

In [ ]:
import yaml

def patch_models_yaml(
    models_path: Path,
    model_key: str,
    model_dir: Path,
    repo_id: str,
) -> None:
    """Patch models.yaml so the vllm model points at our downloaded MODEL_DIR.
    
    Args:
        models_path (Path): The path to the models.yaml file.
        model_key (str): The key of the model to patch.
        model_dir (Path): The directory to download the model to.
        repo_id (str): The ID of the repository to download the model from.
    """
    models = yaml.safe_load(models_path.read_text())
    assert MODEL_KEY in models, f"{MODEL_KEY} not found in models.yaml"

    models[MODEL_KEY]["path"] = str(MODEL_DIR)
    models[MODEL_KEY]["model_name"] = HF_REPO_ID

    models_path.write_text(yaml.safe_dump(models, sort_keys=False))
    print(f"Updated {models_path} for {MODEL_KEY} -> {MODEL_DIR}")

MODELS_PATH = REPO_ROOT / "utility_analysis" / "models.yaml"

patch_models_yaml(MODELS_PATH, MODEL_KEY, MODEL_DIR, HF_REPO_ID)

In [ ]:
# -----------------------------
# User-configurable experiment settings
# -----------------------------
# Keep this equal to MODEL_KEY_FOR_DOWNLOAD (or run the download cell again with a new key).
MODEL_KEY = globals().get("MODEL_KEY_FOR_DOWNLOAD", "llama-31-8b-instruct")

# Inputs
OPTIONS_PATH = "../../shared_options/options_refined.json"
ROLES_CONFIG_PATH = "../../shared_options/role_sets.yaml"
ROLESET = "refined_default"
# Point this to your precomputed utilities JSON aligned by option id.
UTILITIES_PATH = f"../../shared_utilities/options_refined/{MODEL_KEY}/results_utilities_{MODEL_KEY}_software_engineer.json"

# Activation extraction settings
LAYERS = "all"                  # e.g. "all", "0-31", "10-20"
MAX_NEW_TOKENS_FOR_PARSING = 2   # keep generation short

# Probe training settings
POSITION = "gen_first"          # "gen_first" or "prompt_last"
TARGET = "utility"              # "utility" or "rating"
PROBE_MODE = "all"              # "all", "per_role", "cross_role"
TEST_FRACTION = 0.2
RIDGE_LAMBDA = 1.0
SEED = 42

# Output
SAVE_DIR = f"results_linear_probes/{MODEL_KEY}"
SAVE_SUFFIX = MODEL_KEY

print("Configured model:", MODEL_KEY)
print("Collect output dir:", SAVE_DIR)

In [ ]:
# Build an override config for the collect stage.
collect_overrides = {
    "model_key": MODEL_KEY,
    "stage": "collect",
    "save_dir": SAVE_DIR,
    "save_suffix": SAVE_SUFFIX,
    "options_path": OPTIONS_PATH,
    "utilities_path": UTILITIES_PATH,
    "roleset": ROLESET,
    "roles_config_path": ROLES_CONFIG_PATH,
    "layers": LAYERS,
    "max_new_tokens_for_parsing": MAX_NEW_TOKENS_FOR_PARSING,
    "fp16": True,
    "trust_remote_code": True,
}

collect_cfg_path = utility_dir / "linear_probes_collect_override.yaml"
with open(collect_cfg_path, "w") as f:
    yaml.safe_dump(collect_overrides, f, sort_keys=False)

print("Wrote:", collect_cfg_path)
print(yaml.safe_dump(collect_overrides, sort_keys=False))

In [ ]:
# Run activation/rating collection.
cmd = [
    "python3",
    "-u",
    "run_experiments.py",
    "--experiments", "linear_probes_collect",
    "--models", MODEL_KEY,
    "--config", str(collect_cfg_path),
    "--overwrite_results",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=utility_dir)
if proc.returncode != 0:
    raise RuntimeError(f"Collect stage failed with code {proc.returncode}")

In [ ]:
# Build train override config and run probe training.
train_overrides = {
    "model_key": MODEL_KEY,
    "stage": "train",
    "save_dir": SAVE_DIR,
    "save_suffix": SAVE_SUFFIX,
    "position": POSITION,
    "target": TARGET,
    "probe_mode": PROBE_MODE,
    "test_fraction": TEST_FRACTION,
    "ridge_lambda": RIDGE_LAMBDA,
    "seed": SEED,
}

train_cfg_path = utility_dir / "linear_probes_train_override.yaml"
with open(train_cfg_path, "w") as f:
    yaml.safe_dump(train_overrides, f, sort_keys=False)

print("Wrote:", train_cfg_path)
print(yaml.safe_dump(train_overrides, sort_keys=False))

cmd = [
    "python3",
    "-u",
    "run_experiments.py",
    "--experiments", "linear_probes_train",
    "--models", MODEL_KEY,
    "--config", str(train_cfg_path),
    "--overwrite_results",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=utility_dir)
if proc.returncode != 0:
    raise RuntimeError(f"Train stage failed with code {proc.returncode}")

In [ ]:
# Quick results load + layer sweep plot.
import matplotlib.pyplot as plt

results_path = (
    utility_dir
    / SAVE_DIR
    / f"linear_probes_{SAVE_SUFFIX}_probe_results_{POSITION}_{TARGET}_{PROBE_MODE}.json"
)
print("Loading:", results_path)

with open(results_path, "r") as f:
    results = json.load(f)

if "metrics_by_layer" in results:
    metrics = results["metrics_by_layer"]
else:
    # For per_role or cross_role modes, aggregate by averaging over splits.
    bucket = []
    if "by_role" in results:
        bucket = [v["metrics_by_layer"] for v in results["by_role"].values()]
    elif "leave_one_role_out" in results:
        bucket = [v["metrics_by_layer"] for v in results["leave_one_role_out"].values()]

    metrics = {}
    for d in bucket:
        for layer, m in d.items():
            metrics.setdefault(layer, {"r2": [], "mse": [], "spearman": []})
            metrics[layer]["r2"].append(m["r2"])
            metrics[layer]["mse"].append(m["mse"])
            metrics[layer]["spearman"].append(m["spearman"])
    metrics = {
        str(k): {
            "r2": float(sum(v["r2"]) / len(v["r2"])),
            "mse": float(sum(v["mse"]) / len(v["mse"])),
            "spearman": float(sum(v["spearman"]) / len(v["spearman"])),
        }
        for k, v in metrics.items()
    }

layers = sorted(int(k) for k in metrics.keys())
r2_vals = [metrics[str(l)]["r2"] for l in layers]
s_vals = [metrics[str(l)]["spearman"] for l in layers]

print("Top 5 layers by R2:")
for l in sorted(layers, key=lambda x: metrics[str(x)]["r2"], reverse=True)[:5]:
    print(f"layer={l:>3}  r2={metrics[str(l)]['r2']:.4f}  spearman={metrics[str(l)]['spearman']:.4f}")

plt.figure(figsize=(10, 4))
plt.plot(layers, r2_vals, label="R2")
plt.plot(layers, s_vals, label="Spearman")
plt.xlabel("Layer")
plt.ylabel("Score")
plt.title(f"Linear Probe Layer Sweep ({POSITION}, target={TARGET}, mode={PROBE_MODE})")
plt.legend()
plt.grid(alpha=0.2)
plt.show()